# V2.19 — Clinical V2 Fine-Tuning Schedule Comparison

**Issue:** [#160](https://github.com/romanpoluden/revela/issues/160)  
**Branch:** `clinical_v2_16_17_18`  
**Date:** 2026-05-24  
**Author:** Rehma Aziz  
**Scope:** Test two alternative LR schedules — staged fine-tuning and low-LR cosine decay — against the standard full-model EfficientNet-B0 training on the frozen `clinical_v2` test split. Same backbone, same split, same augmentation, same seed. Only the training schedule differs.

---

## Outcome Summary

**Neither schedule variant improves over the standard baseline.** Standard full-model training at lr=1e-4 (baseline-regen: macro-F1 0.6527, lesion FN 70) outperforms both staged fine-tuning (0.6192, FN 95) and low-LR cosine (0.6306, FN 82) on the frozen test set. Low-LR achieved the best val score (0.6698) but showed the largest val/test gap. The schedule axis is not the lever — V2.18's ConvNeXt-Tiny improvement was architectural.

This is an educational model evaluation. The lesion class is a routing output for dermoscopic review, **not** cancer detection. No diagnostic or clinical-readiness claims are made.

## Setup

In [ ]:
import hashlib
import json

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd

ROOT = "../../.."  # relative to this notebook

VARIANTS = ["baseline_regen", "staged_finetune", "low_lr_finetune"]
LABELS = {
    "baseline_regen":   "B0 baseline",
    "staged_finetune":  "Staged fine-tuning",
    "low_lr_finetune":  "Low-LR cosine",
}
COLORS = {
    "baseline_regen":  "#7f8c8d",
    "staged_finetune": "#3498db",
    "low_lr_finetune": "#e67e22",
}

HISTORY = {
    "baseline_regen":  f"{ROOT}/models/clinical_v2_effnet_b0/training_history.csv",
    "staged_finetune": f"{ROOT}/models/clinical_v2_staged_finetune_effnet_b0/training_history.csv",
    "low_lr_finetune": f"{ROOT}/models/clinical_v2_low_lr_finetune_effnet_b0/training_history.csv",
}
METRICS = {
    "baseline_regen":  f"{ROOT}/outputs/metrics/clinical_v2_baseline_regen_test_metrics.json",
    "staged_finetune": f"{ROOT}/outputs/metrics/clinical_v2_staged_finetune_test_metrics.json",
    "low_lr_finetune": f"{ROOT}/outputs/metrics/clinical_v2_low_lr_finetune_test_metrics.json",
}
SOURCE_METRICS = {
    "baseline_regen":  f"{ROOT}/outputs/metrics/clinical_v2_baseline_regen_source_metrics.csv",
    "staged_finetune": f"{ROOT}/outputs/metrics/clinical_v2_staged_finetune_source_metrics.csv",
    "low_lr_finetune": f"{ROOT}/outputs/metrics/clinical_v2_low_lr_finetune_source_metrics.csv",
}
CONFUSION_IMGS = {
    "baseline_regen":  f"{ROOT}/outputs/plots/clinical_v2_baseline_regen_confusion_matrix.png",
    "staged_finetune": f"{ROOT}/outputs/plots/clinical_v2_staged_finetune_confusion_matrix.png",
    "low_lr_finetune": f"{ROOT}/outputs/plots/clinical_v2_low_lr_finetune_confusion_matrix.png",
}
COMPARISON_CSV = f"{ROOT}/outputs/metrics/clinical_v2_finetuning_comparison_table.csv"

# Frozen test-split fingerprint (issue #160).
TEST_PATH = f"{ROOT}/data/processed/clinical_v2/test.csv"
test_hash = hashlib.md5(open(TEST_PATH, "rb").read()).hexdigest()
print("Test hash:", test_hash)
assert test_hash == "4b510381927f6265446a62cb990e69fd", "Test split changed — STOP"
print("Frozen test split verified (1515 rows).")

## Section 1 — Schedule Configuration

All non-schedule settings are identical: EfficientNet-B0, ImageNet pretrained, 224×224, AdamW (weight_decay=0.01), inverse-frequency class weights, no sampler, seed 42, baseline augmentation.

| Variant | Phase 1 | Phase 2 | Total epochs |
|---|---|---|---:|
| Baseline-regen | Full model, lr=1e-4, constant | — | 5 |
| Staged fine-tuning | Head only, lr=1e-3 (backbone frozen) | Full model, lr=5e-5, cosine | 2 + 5 = 7 |
| Low-LR cosine | Full model, lr=5e-5, cosine | — | 8 |

Best checkpoint selected by `val_macro_f1` across all epochs/phases.

## Section 2 — Training Results

In [ ]:
histories = {v: pd.read_csv(p) for v, p in HISTORY.items()}

rows = []
for v, h in histories.items():
    best = h.loc[h["val_macro_f1"].idxmax()]
    phase_label = best.get("phase", "full") if "phase" in h.columns else "full"
    rows.append({
        "Variant": LABELS[v],
        "Best epoch": int(best["epoch"]),
        "Best phase": phase_label,
        "Best val macro-F1": round(best["val_macro_f1"], 4),
        "Best val balanced-acc": round(best["val_balanced_accuracy"], 4),
        "Final train loss": round(h.iloc[-1]["train_loss"], 4),
    })
pd.DataFrame(rows).set_index("Variant")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for v, h in histories.items():
    axes[0].plot(h["epoch"], h["val_macro_f1"], marker="o", label=LABELS[v], color=COLORS[v])
    axes[1].plot(h["epoch"], h["val_balanced_accuracy"], marker="o", label=LABELS[v], color=COLORS[v])
    axes[2].plot(h["epoch"], h["train_loss"], marker="o", linestyle="--", label=LABELS[v], color=COLORS[v])

# Mark staged fine-tuning phase boundary (end of head phase at epoch 2)
for ax in axes:
    ax.axvline(2.5, color=COLORS["staged_finetune"], linestyle=":", linewidth=1, alpha=0.6)

axes[0].axhline(0.6420, color="red", linestyle="--", linewidth=1, label="#153 baseline (0.6420)")
axes[0].set_title("Val Macro-F1 (red = #153 baseline)")
axes[1].set_title("Val Balanced Accuracy")
axes[2].set_title("Train Loss")

for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend(fontsize=8)

fig.text(0.5, -0.02, "Dotted vertical line = end of staged fine-tuning head phase (epoch 2→3 boundary)",
         ha="center", fontsize=9, color="grey")
plt.tight_layout()
plt.show()

## Section 3 — Test Set Evaluation

Frozen test set: 1,515 examples. Hash verified above and by the orchestration script after every eval run.

In [ ]:
metrics = {v: json.load(open(p)) for v, p in METRICS.items()}

rows = []
for v, m in metrics.items():
    lr = m["lesion_routing_class"]
    cm = m["confusion_matrix"]
    lesion_fn = sum(cm[4]) - cm[4][4]  # total lesion row minus TP
    rows.append({
        "Variant": LABELS[v],
        "Test accuracy": round(m["test_accuracy"], 4),
        "Macro-F1": round(m["macro_f1"], 4),
        "Balanced acc": round(m["balanced_accuracy"], 4),
        "Lesion recall": round(lr["recall"], 4),
        "Lesion F1": round(lr["f1"], 4),
        "Lesion FN": lesion_fn,
    })
pd.DataFrame(rows).set_index("Variant")

In [ ]:
# Class-wise F1
class_rows = []
for v, m in metrics.items():
    row = {"Variant": LABELS[v]}
    for c in m["class_report"]:
        name = c["class_name"]
        short = name[:32] + "\u2026" if len(name) > 32 else name
        row[short] = round(c["f1"], 4)
    class_rows.append(row)
pd.DataFrame(class_rows).set_index("Variant").T

## Section 4 — Source-Specific Breakdown (SCIN vs Fitzpatrick17k)

In [ ]:
source_dfs = {v: pd.read_csv(p) for v, p in SOURCE_METRICS.items()}

summary = []
for v, df in source_dfs.items():
    for _, r in df.iterrows():
        summary.append({
            "Variant": LABELS[v],
            "Source": r["source_dataset"],
            "n": int(r["support"]),
            "Accuracy": round(r["accuracy"], 4),
            "Macro-F1": round(r["macro_f1"], 4),
            "Balanced acc": round(r["balanced_accuracy"], 4),
        })

src_df = pd.DataFrame(summary)
src_df[src_df["Source"] != "combined"].set_index(["Source", "Variant"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
baseline_regen_scin = src_df[(src_df["Source"] == "google_scin") & (src_df["Variant"] == "B0 baseline")]["Macro-F1"].values[0]
baseline_regen_fitz = src_df[(src_df["Source"] == "fitzpatrick17k") & (src_df["Variant"] == "B0 baseline")]["Macro-F1"].values[0]

for ax, source, thr in [
    (axes[0], "google_scin", baseline_regen_scin),
    (axes[1], "fitzpatrick17k", baseline_regen_fitz),
]:
    subset = src_df[src_df["Source"] == source]
    bars = ax.bar(subset["Variant"], subset["Macro-F1"],
                  color=[COLORS[v] for v in VARIANTS])
    ax.axhline(thr, color="red", linestyle="--", linewidth=1, label=f"baseline {thr:.4f}")
    ax.set_title(f"{source} macro-F1")
    ax.set_ylim(0, max(subset["Macro-F1"]) * 1.15)
    ax.legend(fontsize=8)
    for bar, val in zip(bars, subset["Macro-F1"]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{val:.4f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

## Section 5 — Full Comparison Table

Deltas are vs the same-run baseline-regen (not the original #153 reference).

In [ ]:
comp = pd.read_csv(COMPARISON_CSV).set_index("metric")
comp

## Section 6 — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, v in zip(axes, VARIANTS):
    ax.imshow(mpimg.imread(CONFUSION_IMGS[v]))
    ax.set_title(LABELS[v], fontsize=12, fontweight="bold")
    ax.axis("off")
plt.suptitle("Confusion Matrices — Frozen Test Set (n=1,515)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Section 7 — Verdict

### Baseline-regen: REFERENCE
| Metric | Value |
|---|---|
| Test macro-F1 | 0.6527 (+0.0107 vs #153 ref) |
| Balanced accuracy | 0.6612 (+0.0041 vs #153 ref) |
| Lesion routing FN | 70 (below ≤76 threshold) |

### Staged fine-tuning: NOT ADOPTED
| Criterion | Baseline-regen | Staged | Pass |
|---|---|---|---|
| Test macro-F1 | 0.6527 | 0.6192 (−0.0335) | ✗ |
| Balanced accuracy | 0.6612 | 0.6352 (−0.0260) | ✗ |
| SCIN macro-F1 | 0.4267 | 0.3919 (−0.0348) | ✗ |
| Lesion routing FN | 70 | 95 (+25) | ✗ |

Head-only warm-up phase gives the backbone a weaker gradient signal before unfreezing. Lesion-routing FN jumps from 70 to 95 — well above the ≤76 promotion threshold from #153.

### Low-LR cosine: NOT ADOPTED
| Criterion | Baseline-regen | Low-LR | Pass |
|---|---|---|---|
| Test macro-F1 | 0.6527 | 0.6306 (−0.0221) | ✗ |
| Balanced accuracy | 0.6612 | 0.6463 (−0.0149) | ✗ |
| SCIN macro-F1 | 0.4267 | 0.3826 (−0.0441) | ✗ |
| Lesion routing FN | 70 | 82 (+12) | ✗ |

Achieved the best val macro-F1 (0.6698) of all three variants but has the widest val/test gap — 8 epochs at a lower LR overfits to the validation distribution without improving generalisation.

### Key finding
The schedule axis does not explain V2.18's ConvNeXt-Tiny improvement. That was architectural capacity (+23.8M params). For EfficientNet-B0 on `clinical_v2`, the default schedule (1e-4 constant, 5 epochs) is already at or near optimum. Future B0 experiments should focus on data or augmentation strategy rather than schedule tuning.

## Summary

| Item | Value |
|---|---|
| Task | V2.19 — fine-tuning schedule comparison (#160) |
| Variants trained | Baseline-regen, Staged fine-tuning (2+5 ep), Low-LR cosine (8 ep) |
| Hardware | Apple Silicon MPS; caffeinate to prevent idle-sleep throttling |
| Train / Val / Test | 6,986 / 1,518 / 1,515 (frozen) |
| Test hash | `4b510381927f6265446a62cb990e69fd` ✓ |
| Baseline-regen test macro-F1 | 0.6527 (lesion FN 70) |
| Staged fine-tuning test macro-F1 | 0.6192 (lesion FN 95) — NOT ADOPTED |
| Low-LR cosine test macro-F1 | 0.6306 (lesion FN 82) — NOT ADOPTED |
| Recommendation | Schedule is not the lever; pursue architecture or data next |

Out of scope confirmed: taxonomy unchanged, inference registry unchanged, no app wiring touched, no clinical-readiness claims, no model promoted in code.

**Artifacts:** `docs/model/clinical_v2_staged_finetuning_summary.md`, `outputs/metrics/clinical_v2_finetuning_comparison_table.csv`, `outputs/plots/clinical_v2_*_confusion_matrix.png`.